# Fine-tuning BGE-M3 V2: Chapter Split + Hard Negative Mining

Notebook này fine-tune `BAAI/bge-m3` theo cùng quy trình với Vietnamese Bi-Encoder V2:

- Đọc dữ liệu QA retrieval từ 3 tập canonical: `training_data_lsd`, `training_data_pldc`, `training_data_th`.
- Tách `train / val / test` theo `(source, chapter)` để giảm leakage.
- Đánh giá baseline BGE-M3 trên validation và test.
- Train vòng 1 bằng `MultipleNegativesRankingLoss`.
- Mine hard negatives bằng model vòng 1.
- Train vòng 2 với mined hard negatives.
- Xuất model, metadata và bảng metrics để so sánh với notebook Bi-Encoder cũ.

Khuyến nghị Kaggle: bật GPU T4/P100, bật Internet, add dataset `dangvy1507/training-data`. BGE-M3 lớn hơn Vietnamese Bi-Encoder nên batch size mặc định nhỏ hơn.

In [ ]:
# %pip install -U "sentence-transformers>=3.0.0" datasets torch accelerate --quiet

## 1. Import và cấu hình

In [ ]:
import json
import math
import random
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import MultipleNegativesRankingLoss

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

MODEL_NAME = "BAAI/bge-m3"
CANONICAL_DIRS = ["training_data_lsd", "training_data_pldc", "training_data_th"]
KAGGLE_TRAINING_BASE = Path("/kaggle/input/datasets/dangvy1507/training-data")

if Path("/kaggle").is_dir():
    REPO_ROOT = Path("/kaggle/working")
    TRAINING_BASE = KAGGLE_TRAINING_BASE
    print("Environment: Kaggle")
else:
    REPO_ROOT = Path.cwd()
    TRAINING_BASE = REPO_ROOT / "data" / "training_data"
    print("Environment: local")

AUDIT_FILES = [TRAINING_BASE / name / "qa_audit.jsonl" for name in CANONICAL_DIRS]
AUDIT_FILES = [p for p in AUDIT_FILES if p.is_file()]
if not AUDIT_FILES:
    raise FileNotFoundError(f"Không tìm thấy qa_audit.jsonl canonical dưới {TRAINING_BASE}")

OUTPUT_BASE = REPO_ROOT / "models" / "bge_m3_hnm_v2"
CKPT_V1 = OUTPUT_BASE / "checkpoints_v1"
CKPT_V2 = OUTPUT_BASE / "checkpoints_v2"
MODEL_V1_DIR = OUTPUT_BASE / "bge-m3-v1"
MODEL_V2_DIR = OUTPUT_BASE / "bge-m3-v2-hnm"
ARTIFACT_DIR = OUTPUT_BASE / "artifacts"
for d in [CKPT_V1, CKPT_V2, MODEL_V1_DIR, MODEL_V2_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

EPOCHS_V1 = 3
LR_V1 = 1e-5
EPOCHS_V2 = 2
LR_V2 = 5e-6

# BGE-M3 lớn hơn model bkai. Cấu hình dưới đây ưu tiên tránh OOM trên Kaggle T4 16GB.
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
MINE_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 16
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
EVAL_STEPS = 100
SAVE_STEPS = 100
LOGGING_STEPS = 20
MINE_TOP_K = 30
USE_GRADIENT_CHECKPOINTING = True
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nAudit files:")
for p in AUDIT_FILES:
    print(" -", p)
print(f"\nOutput: {OUTPUT_BASE}")

## 2. Load dữ liệu và split theo chương

In [ ]:
def load_jsonl(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_all_audits(paths):
    rows = []
    for path in paths:
        source = path.parent.name
        for row in load_jsonl(path):
            row["_source"] = source
            rows.append(row)
    return rows


def chapter_key(row):
    return (row.get("_source", ""), row.get("chapter", "Unknown"))


def split_by_chapter_3way(rows, val_ratio=0.15, test_ratio=0.15, seed=42):
    by_key = defaultdict(list)
    for row in rows:
        by_key[chapter_key(row)].append(row)

    keys = list(by_key.keys())
    rng = random.Random(seed)
    rng.shuffle(keys)

    n_total = len(rows)
    val_target = int(n_total * val_ratio)
    test_target = int(n_total * test_ratio)

    train_rows, val_rows, test_rows = [], [], []
    for key in keys:
        bucket = by_key[key]
        if len(val_rows) < val_target:
            val_rows.extend(bucket)
        elif len(test_rows) < test_target:
            test_rows.extend(bucket)
        else:
            train_rows.extend(bucket)
    return train_rows, val_rows, test_rows


all_pairs = load_all_audits(AUDIT_FILES)
train_pairs, val_pairs, test_pairs = split_by_chapter_3way(all_pairs, VAL_RATIO, TEST_RATIO, RANDOM_SEED)

print(f"Total pairs: {len(all_pairs)}")
print(f"Train      : {len(train_pairs)} ({len(train_pairs)/len(all_pairs)*100:.1f}%)")
print(f"Val        : {len(val_pairs)} ({len(val_pairs)/len(all_pairs)*100:.1f}%)")
print(f"Test       : {len(test_pairs)} ({len(test_pairs)/len(all_pairs)*100:.1f}%)")

split_keys = {
    "train": set(chapter_key(r) for r in train_pairs),
    "val": set(chapter_key(r) for r in val_pairs),
    "test": set(chapter_key(r) for r in test_pairs),
}
print("\nChapter-key overlap checks:")
print(" train & val :", len(split_keys["train"] & split_keys["val"]))
print(" train & test:", len(split_keys["train"] & split_keys["test"]))
print(" val & test  :", len(split_keys["val"] & split_keys["test"]))

print("\nIntent distribution:")
for name, rows in [("train", train_pairs), ("val", val_pairs), ("test", test_pairs)]:
    print(name, Counter(r.get("intent_type", "unknown") for r in rows))

s = train_pairs[0]
print("\nSample query   :", s.get("query", "")[:200])
print("Sample positive:", s.get("positive", "")[:200])

## 3. Dataset helpers và evaluator

In [ ]:
def clean_text(text):
    return " ".join(str(text or "").split())


def pairs_to_dataset(rows, use_negative=False):
    records = []
    for row in rows:
        query = clean_text(row.get("query"))
        positive = clean_text(row.get("positive"))
        if not query or not positive:
            continue
        record = {"anchor": query, "positive": positive}
        if use_negative:
            negative = clean_text(row.get("mined_negative") or row.get("negative"))
            if negative:
                record["negative"] = negative
        records.append(record)
    return Dataset.from_list(records)


def make_ir_eval(rows, all_rows, name):
    queries = {}
    corpus = {}
    relevant_docs = {}

    for i, row in enumerate(rows):
        qid = f"{name}_q_{i}"
        pid = f"{name}_p_{i}"
        queries[qid] = clean_text(row["query"])
        corpus[pid] = clean_text(row["positive"])
        relevant_docs[qid] = {pid}

    existing = set(corpus.values())
    for j, row in enumerate(all_rows):
        text = clean_text(row.get("positive"))
        if text and text not in existing:
            corpus[f"corpus_{j}"] = text
            existing.add(text)

    evaluator = InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        name=name,
        precision_recall_at_k=[1, 5, 10],
        mrr_at_k=[10],
        show_progress_bar=False,
    )
    return evaluator, queries, corpus, relevant_docs


def evaluator_main_score(result):
    if isinstance(result, dict):
        for key, value in result.items():
            if "mrr@10" in key.lower():
                return float(value)
        return float(next(iter(result.values())))
    return float(result)


def metric_key_for_best(result):
    if isinstance(result, dict):
        for key in result:
            if "mrr@10" in key.lower():
                return f"eval_{key}"
    return "eval_val_cosine_mrr@10"


def flatten_metrics(prefix, result):
    if not isinstance(result, dict):
        return {f"{prefix}_mrr@10": float(result)}
    return {f"{prefix}_{key}": float(value) for key, value in result.items()}


train_dataset_v1 = pairs_to_dataset(train_pairs, use_negative=False)
val_dataset = pairs_to_dataset(val_pairs, use_negative=False)
test_dataset = pairs_to_dataset(test_pairs, use_negative=False)

val_evaluator, val_queries, val_corpus, val_relevant = make_ir_eval(val_pairs, all_pairs, "val")
test_evaluator, test_queries, test_corpus, test_relevant = make_ir_eval(test_pairs, all_pairs, "test")

print(train_dataset_v1)
print(val_dataset)
print(test_dataset)
print(f"Val evaluator : queries={len(val_queries)}, corpus={len(val_corpus)}")
print(f"Test evaluator: queries={len(test_queries)}, corpus={len(test_corpus)}")

## 4. Baseline BGE-M3

In [ ]:
base_model = SentenceTransformer(MODEL_NAME)
print(f"Loaded base model: {MODEL_NAME}")
print(f"Embedding dim: {base_model.get_sentence_embedding_dimension()}")
print(f"Device: {base_model.device}")

print("\nEvaluating base BGE-M3 on val...")
base_val_result = val_evaluator(base_model, output_path=None)
base_val_mrr = evaluator_main_score(base_val_result)
METRIC_FOR_BEST = metric_key_for_best(base_val_result)

print("Evaluating base BGE-M3 on test...")
base_test_result = test_evaluator(base_model, output_path=None)
base_test_mrr = evaluator_main_score(base_test_result)

print(f"Base val  MRR@10: {base_val_mrr:.4f}")
print(f"Base test MRR@10: {base_test_mrr:.4f}")
print(f"metric_for_best_model: {METRIC_FOR_BEST}")

## 5. Train vòng 1 bằng MultipleNegativesRankingLoss

In [ ]:
def train_sentence_transformer(model, train_dataset, eval_dataset, evaluator, output_dir, epochs, lr, run_name):
    loss = MultipleNegativesRankingLoss(model)
    steps_per_epoch = max(1, math.ceil(len(train_dataset) / BATCH_SIZE / GRADIENT_ACCUMULATION_STEPS))
    total_steps = steps_per_epoch * epochs
    warmup_steps = int(total_steps * WARMUP_RATIO)

    print(f"[{run_name}] train samples: {len(train_dataset)}")
    print(f"[{run_name}] batch size   : {BATCH_SIZE}")
    print(f"[{run_name}] grad accum   : {GRADIENT_ACCUMULATION_STEPS}")
    print(f"[{run_name}] steps/epoch  : {steps_per_epoch}")
    print(f"[{run_name}] total steps  : {total_steps}")
    print(f"[{run_name}] warmup       : {warmup_steps}")

    args = SentenceTransformerTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model=METRIC_FOR_BEST,
        greater_is_better=True,
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        logging_first_step=True,
        logging_dir=str(output_dir / "logs"),
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
        seed=RANDOM_SEED,
        disable_tqdm=False,
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=loss,
        evaluator=evaluator,
    )
    trainer.train()
    return model, trainer


model_v1, trainer_v1 = train_sentence_transformer(
    base_model,
    train_dataset_v1,
    val_dataset,
    val_evaluator,
    CKPT_V1,
    EPOCHS_V1,
    LR_V1,
    "v1",
)

## 6. Đánh giá vòng 1

In [ ]:
print("Evaluating v1 on val...")
v1_val_result = val_evaluator(model_v1, output_path=None)
v1_val_mrr = evaluator_main_score(v1_val_result)

print("Evaluating v1 on test...")
v1_test_result = test_evaluator(model_v1, output_path=None)
v1_test_mrr = evaluator_main_score(v1_test_result)

print(f"V1 val  MRR@10: {v1_val_mrr:.4f}")
print(f"V1 test MRR@10: {v1_test_mrr:.4f}")

## 7. Hard Negative Mining

Với mỗi query trong train set, dùng model vòng 1 retrieve top-k trong corpus train. Chọn đoạn sai có score cao nhất làm `mined_negative`.

In [ ]:
def mine_hard_negatives(model, train_rows, top_k=30, batch_size=16):
    corpus_texts = []
    text_to_idx = {}
    for row in train_rows:
        text = clean_text(row.get("positive"))
        if text and text not in text_to_idx:
            text_to_idx[text] = len(corpus_texts)
            corpus_texts.append(text)

    queries = [clean_text(row.get("query")) for row in train_rows]
    positive_texts = [clean_text(row.get("positive")) for row in train_rows]

    print(f"Mining corpus size: {len(corpus_texts)}")
    print(f"Mining queries    : {len(queries)}")

    t0 = time.time()
    corpus_emb = model.encode(
        corpus_texts,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=batch_size,
    )
    query_emb = model.encode(
        queries,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=batch_size,
    )

    mined_rows = []
    no_negative_count = 0
    for i, row in enumerate(train_rows):
        scores = query_emb[i] @ corpus_emb.T
        k = min(top_k, len(corpus_texts))
        top_idx = np.argpartition(-scores, range(k))[:k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        pos_text = positive_texts[i]
        neg_text = None
        neg_score = None
        for idx in top_idx:
            candidate = corpus_texts[int(idx)]
            if candidate == pos_text:
                continue
            neg_text = candidate
            neg_score = float(scores[int(idx)])
            break

        if neg_text is None:
            existing_negs = row.get("negatives") or []
            neg_text = clean_text(existing_negs[0]) if existing_negs else None
            no_negative_count += 1

        new_row = dict(row)
        if neg_text:
            new_row["mined_negative"] = neg_text
            new_row["mined_negative_score"] = neg_score
        mined_rows.append(new_row)

    print(f"Mining done in {time.time() - t0:.1f}s")
    print(f"Fallback no-negative count: {no_negative_count}")
    return mined_rows


train_pairs_mined = mine_hard_negatives(model_v1, train_pairs, MINE_TOP_K, MINE_BATCH_SIZE)

mined_path = ARTIFACT_DIR / "train_pairs_with_mined_negatives.jsonl"
with open(mined_path, "w", encoding="utf-8") as f:
    for row in train_pairs_mined:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Saved mined negatives: {mined_path}")

train_dataset_v2 = pairs_to_dataset(train_pairs_mined, use_negative=True)
print(train_dataset_v2)
print(train_dataset_v2[0].keys())
print("Sample mined negative:", train_dataset_v2[0].get("negative", "")[:200])

## 8. Train vòng 2 với mined hard negatives

In [ ]:
model_v2, trainer_v2 = train_sentence_transformer(
    model_v1,
    train_dataset_v2,
    val_dataset,
    val_evaluator,
    CKPT_V2,
    EPOCHS_V2,
    LR_V2,
    "v2_hnm",
)

## 9. Final evaluation trên test set độc lập

In [ ]:
print("Evaluating v2 on val...")
v2_val_result = val_evaluator(model_v2, output_path=None)
v2_val_mrr = evaluator_main_score(v2_val_result)

print("Evaluating v2 on test...")
v2_test_result = test_evaluator(model_v2, output_path=None)
v2_test_mrr = evaluator_main_score(v2_test_result)

summary_rows = [
    {"model": "bge_m3_base", "val_mrr@10": base_val_mrr, "test_mrr@10": base_test_mrr},
    {"model": "bge_m3_v1_mnrl", "val_mrr@10": v1_val_mrr, "test_mrr@10": v1_test_mrr},
    {"model": "bge_m3_v2_hard_negative", "val_mrr@10": v2_val_mrr, "test_mrr@10": v2_test_mrr},
]

print("\nMRR@10 summary")
print("=" * 84)
print(f"{'model':<30} {'val_mrr@10':>12} {'test_mrr@10':>13} {'test_delta_base':>16}")
for row in summary_rows:
    delta = row["test_mrr@10"] - base_test_mrr
    print(f"{row['model']:<30} {row['val_mrr@10']:>12.4f} {row['test_mrr@10']:>13.4f} {delta:>+16.4f}")

full_metrics = {
    "base_val": flatten_metrics("base_val", base_val_result),
    "base_test": flatten_metrics("base_test", base_test_result),
    "v1_val": flatten_metrics("v1_val", v1_val_result),
    "v1_test": flatten_metrics("v1_test", v1_test_result),
    "v2_val": flatten_metrics("v2_val", v2_val_result),
    "v2_test": flatten_metrics("v2_test", v2_test_result),
}

with open(ARTIFACT_DIR / "bge_m3_hnm_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump({"summary": summary_rows, "metrics": full_metrics}, f, ensure_ascii=False, indent=2)

csv_path = ARTIFACT_DIR / "bge_m3_hnm_eval_summary.csv"
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("model,val_mrr@10,test_mrr@10,test_delta_base\n")
    for row in summary_rows:
        delta = row["test_mrr@10"] - base_test_mrr
        f.write(f"{row['model']},{row['val_mrr@10']:.6f},{row['test_mrr@10']:.6f},{delta:.6f}\n")

print(f"\nSaved metrics: {ARTIFACT_DIR / 'bge_m3_hnm_eval_summary.json'}")
print(f"Saved CSV    : {csv_path}")

## 10. Save models và metadata

In [ ]:
model_v1.save(str(MODEL_V1_DIR))
model_v2.save(str(MODEL_V2_DIR))

metadata = {
    "base_model": MODEL_NAME,
    "audit_files": [str(p) for p in AUDIT_FILES],
    "canonical_dirs": CANONICAL_DIRS,
    "split": {
        "train_pairs": len(train_pairs),
        "val_pairs": len(val_pairs),
        "test_pairs": len(test_pairs),
        "split_unit": "(source, chapter)",
        "seed": RANDOM_SEED,
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
    },
    "stage_1": {
        "epochs": EPOCHS_V1,
        "lr": LR_V1,
        "loss": "MultipleNegativesRankingLoss",
    },
    "stage_2": {
        "epochs": EPOCHS_V2,
        "lr": LR_V2,
        "loss": "MultipleNegativesRankingLoss with mined hard negatives",
        "mine_top_k": MINE_TOP_K,
    },
    "mrr@10": {
        "base_val": base_val_mrr,
        "base_test": base_test_mrr,
        "v1_val": v1_val_mrr,
        "v1_test": v1_test_mrr,
        "v2_val": v2_val_mrr,
        "v2_test": v2_test_mrr,
    },
}

for out_dir in [MODEL_V1_DIR, MODEL_V2_DIR]:
    with open(out_dir / "training_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Saved V1 model: {MODEL_V1_DIR}")
print(f"Saved V2 model: {MODEL_V2_DIR}")
print("\nUse V2 model in RAG:")
print("from sentence_transformers import SentenceTransformer")
print(f"model = SentenceTransformer('{MODEL_V2_DIR.resolve()}')")